# Linear (Uniform) Post-Training Quantisation
Map weights or activations to a discrete set of integers
$x ∈ [x_{min}, x_{max}] → x_q \in [0, 2^b - 1]$ where $b$ = bit-width:


In [1]:
import math
import torch
from torch import Tensor

**1. Define Scale $s$ (width of each quantisation bucket):**
$$s = \frac{x_{max} - x_{min}}{2^b}$$

**2. Define Zero-point Alignment $z$ to ensure $x_{min}$ maps to 0:**
$$z = -\frac{x_{min}}{s} = x_{min}$$

**3. Quantization (Forward):**
$$x_{q} = \text{clamp}\left( \left\lfloor \frac{x}{s} \right\rfloor + z, 0, 2^{b}-1 \right)$$

**4. Dequantization (Reverse):**
$$x_{dq} = s \cdot (x_q - z) ≈ x$$

> **Convention note:** the paper writes $x_{dq} = s \cdot (x_q - z)$ with an integer zero-point.
> Here we store $z = x_{min}$ as a float and absorb it into the dequantisation offset,
> which is algebraically equivalent and simpler to implement in PTQ.

In [4]:
def compute_scale_zeropoint(x_min: Tensor, x_max: Tensor, bit_width: int,) -> tuple[Tensor, Tensor]:
  """
  Compute quantisation scale and zero-point

    s = (max_x - min_x) / (2^b)
    z = min_x

  Parameters
  ----------
  x_min, x_max : Tensor - observed minimum and maximum values (scalar or batched)
  bit_width : int - Target quantization bit-width b

  Returns
  -------
  scale      : Tensor  (same shape as x_min / x_max)
  zero_point : Tensor  (same shape, stored as float for arithmetic convenience)
  """
  assert bit_width >= 1, "bit_width must be at least 1"
  n_levels = 2 ** bit_width          # number of quantization levels = 2^b

  # Clamp to avoid division by zero for constant tensors
  range_ = (x_max - x_min).clamp(min=1e-8)
  scale      = range_ / n_levels      # s
  zero_point = x_min                  # z  (float, not rounded to int)

  return scale, zero_point


In [9]:
def linearQuantise(x: Tensor, scale: Tensor, zero_point: Tensor, bit_width: int) -> Tensor:
  """
  Quantise x to integer codes in [0, 2^b - 1]

    x_q = clamp( round(x - s) + z,  0,  2^b - 1 )

  Parameters
  ----------
  x          : Tensor  — floating-point input
  scale      : Tensor  — broadcastable against x
  zero_point : Tensor  — broadcastable against x  (float)
  bit_width  : int

  Returns
  -------
  x_q : Tensor  — integer-valued tensor (dtype float for downstream ops)

  """
  q_min = 0
  q_max = 2 ** bit_width - 1

  # Shift and scale, round to nearest integer, clamp to valid grid
  x_q = torch.clamp(torch.round(x - zero_point) / scale, min=q_min, max=q_max)
  return x_q

In [10]:
def linearDequantise(x_q: Tensor, scale: Tensor, zero_point: Tensor) -> Tensor:
  """
  Dequantise x_q to floating-point values

    x_dq = s * (x_q - z)

  Parameters
  ----------
  x_q        : Tensor  — quantized integer codes (stored as float32)
  scale      : Tensor  — broadcastable against x_q
  zero_point : Tensor  — broadcastable against x_q (float)

  Returns
  -------
  x_dq : Tensor  — reconstructed floating-point values
  """
  return scale * x_q + zero_point

In [14]:
def fake_quantise(x: Tensor, scale: Tensor, zero_point: Tensor, bit_width: int) -> Tensor:
  """
  Simulate quantisation in floating-point (fake-quant)

  Applies linearQuantise then linearDequantise so the tensor stays in float32 but carries the quantisation error.
  This is the operation inserted into the forward pass during calibration.

  Parameters
  ----------
  x          : Tensor  — input activations or weights
  scale      : Tensor  — broadcastable scale
  zero_point : Tensor  — broadcastable zero-point
  bit_width  : int

  Returns
  -------
  x_dq : Tensor  — fake-quantized tensor, same shape and dtype as x
  """
  x_q = linearQuantise(x, scale, zero_point, bit_width)
  x_dq = linearDequantise(x_q, scale, zero_point)
  return x_dq

In [18]:
class LinearQuantiser(torch.nn.Module):
    """
    Stateful linear quantizer for PTQ.

    Calibrate once with set_params(), then call forward() to fake-quantise.
    Scale and zero_point are registered as buffers (not parameters) because pure PTQ does not update them via gradients.

    Attributes
    ----------
    bit_width  : int
    scale      : Tensor (buffer) — fixed after calibration
    zero_point : Tensor (buffer) — fixed after calibration
    """

    def __init__(self, bit_width: int):
        super().__init__()
        self.bit_width = bit_width
        self.register_buffer("scale",      torch.zeros(1))
        self.register_buffer("zero_point", torch.zeros(1))
        self._calibrated: bool = False

    def set_params(self, scale: Tensor, zero_point: Tensor) -> None:
        """
        Store calibrated scale and zero-point in the registered buffers.
        Uses .data.copy_() so the buffer stays a buffer (not replaced).
        """
        self.scale.data      = scale.clone().float().reshape(self.scale.shape)
        self.zero_point.data = zero_point.clone().float().reshape(self.zero_point.shape)
        self._calibrated     = True

    @property
    def is_calibrated(self) -> bool:
        return self._calibrated

    def forward(self, x: Tensor) -> Tensor:
        """
        Fake-quantise x using the calibrated scale and zero-point.

        FIX: removed use_ste parameter — fake_quantise no longer accepts it.
        STE support will be added in the BRECQ module.
        """
        if not self._calibrated:
            raise RuntimeError(
                "LinearQuantizer.forward() called before calibration. "
                "Call set_params(scale, zero_point) first."
            )

        return fake_quantise(x, self.scale, self.zero_point, self.bit_width)

    def extra_repr(self) -> str:
        s = self.scale.item()   if self.scale.numel()      == 1 else "(...)"
        z = self.zero_point.item() if self.zero_point.numel() == 1 else "(...)"
        return f"bit_width={self.bit_width}, calibrated={self._calibrated}, scale={s:.5f}, zero_point={z:.5f}"

## End-to-end test

In [19]:
# Simulate a calibration activation tensor
torch.manual_seed(0)
x_cal = torch.randn(4, 16, 64)   # (batch, pixels, channels)

# Calibrate
s, z = compute_scale_zeropoint(x_cal.min(), x_cal.max(), bit_width=8)
q    = LinearQuantiser(bit_width=8)
q.set_params(s, z)

# Fake-quantise
x_dq = q(x_cal)

mae = (x_dq - x_cal).abs().mean().item()
print(f"Module calibrated: {q.is_calibrated}")
print(f"Mean absolute error: {mae:.6f}")
print(f"Expected MAE ≈ scale/2 = {s.item()/2:.6f}")
print(q)

Module calibrated: True
Mean absolute error: 0.247458
Expected MAE ≈ scale/2 = 0.016006
LinearQuantiser(bit_width=8, calibrated=True, scale=0.03201, zero_point=-4.09374)
